# Week 11 — 멀티모달 Agent (Option A)

> 명세: `docs/WEEK11_TASKS.md` §2~§4. 구현: `src/agent_tools.py`(tool 4종), `src/mm_agent.py`(LangGraph), `src/week11_scenarios.py`(시나리오 러너).
> 실행 로그: `data/week11_scenarios.json`. 아키텍처 문서: `docs/week11_architecture.md`.
>
> 이 노트북은 (1) tool 단독 호출로 입출력 스키마·실패 신호를 시연하고, (2) agent 그래프 구조를 그리고, (3) 시나리오 실행 결과를 표로 보여준다.
> API 비용이 드는 셀은 `RUN_LIVE` 플래그로 막아 두었다 — 시나리오 재실행은 `python -m w11.week11_scenarios`가 정본.

In [1]:
import json
import os
from pathlib import Path

# repo 루트에서 실행 (src/, data/ 상대경로 일치)
if Path.cwd().name == "w11":
    os.chdir("..")
print(Path.cwd())

RUN_LIVE = False  # True로 바꾸면 OpenAI 호출이 있는 셀이 실행됨

/Users/joyoungha/Desktop/project/rag-agent-portfolio


## 1. Tool 단독 호출 — 스키마와 실패 신호 (§9 검증)

각 tool은 단일 책임 + 고정 스키마 + 명시적 실패 신호(`ok=False`)를 지킨다.
오프라인 검증 전체는 `tests/test_week11_agent.py` (14 tests).

In [2]:
from src.agent_tools import ocr_tool

# (a) 텍스트가 많은 페이지 — OCR 성공 (conf ≈ 0.89)
print("text page :", {k: (v[:60] if k == "text" else v) for k, v in
                      ocr_tool("data/sample_images_150/waterpurifier_complex/p029.png").items()})

# (b) 도면-only 페이지 — OCR은 돌았지만 읽을 텍스트가 없음 (ok=True, conf=0.0 = '결과')
print("line-art  :", ocr_tool("data/sample_images_150/waterpurifier_simple/p008.png"))

# (c) 없는 파일 — 명시적 실패 신호 (ok=False + error)
print("missing   :", ocr_tool("data/does_not_exist.png"))

text page : {'text': '관 리 하기 29 5 필 터 가 완전히 장 착 되었는지 확 인 하세요. - 교체 주 기 :12 개 월 (10', 'confidence': 0.891, 'ok': True}
line-art  : {'text': '', 'confidence': 0.0, 'ok': True}
missing   : {'text': '', 'confidence': 0.0, 'ok': False, 'error': "[Errno 2] No such file or directory: 'data/does_not_exist.png'"}


In [3]:
# image_analysis_tool — gpt-4o vision, bbox crop 인자는 10주차 region-crop과 결합용
from src.agent_tools import image_analysis_tool

if RUN_LIVE:
    result = image_analysis_tool(
        "data/sample_images_150/airpurifier_complex/p018.png",
        "조작부는 제품 앞면의 어느 위치에 있나요?",
    )
    print(result)
else:
    # 실패 신호만 오프라인 시연: API 호출 전에 파일 검증이 먼저 실패한다
    print(image_analysis_tool("data/does_not_exist.png", "질문"))

{'image_summary': '', 'confidence': 0.0, 'ok': False, 'error': "[Errno 2] No such file or directory: 'data/does_not_exist.png'"}


In [4]:
# rag_search_tool + answer_generation_tool — 무거운 리소스는 팩토리로 주입
if RUN_LIVE:
    from langchain_openai import ChatOpenAI
    from src.agent_tools import make_rag_search_tool, make_answer_generation_tool
    from src.mm_retrieval import ModalityAwareRetriever
    from src.multimodal import load_mm_store
    from src.retrieval import create_reranker

    reranker = create_reranker()
    retriever = ModalityAwareRetriever(load_mm_store(), reranker)
    rag = make_rag_search_tool(retriever, reranker)

    hit = rag("정수기 필터 교체 주기")
    print("caption_hit:", hit["caption_hit"], "| scores:", [round(s, 3) for s in hit["scores"]])
    for doc in hit["docs"]:
        m = doc.metadata
        print(f"  [{m['category']}_{m['complexity']} p.{m['page']}] ({m.get('modality')})")

## 2. Agent 그래프 — 라우팅과 fallback

- **F1** OCR 실패/저신뢰(conf < 0.5 또는 인식 글자 < 30) → 입력 이미지를 Vision으로 직접 읽기
- **F2** 근거 부족 → **검색된 페이지 이미지**를 질의 시점에 Vision으로 읽기 (Week 9 병목의 질의측 우회).
  트리거 2개: `grade(not_relevant)` **또는** `generate(ungrounded)` — 1차 실행에서 grader 단독 트리거가 너무 관대함을 확인하고 추가
- **F3** 에스컬레이션 후에도 근거 부족 → 명시적 거절 (지어내지 않음)

In [5]:
# 그래프 구조는 stub tool로도 그릴 수 있다 (네트워크·모델 불필요)
from src.mm_agent import build_mm_agent

stub = lambda *a, **k: {}
graph = build_mm_agent(stub, stub, stub, stub, llm=None)
print(graph.get_graph().draw_mermaid())

/Users/joyoungha/Desktop/project/rag-agent-portfolio/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	ocr(ocr)
	vision_input(vision_input)
	rag(rag)
	grade(grade)
	vision_escalate(vision_escalate)
	generate(generate)
	refuse(refuse)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> ocr;
	__start__ -.-> rag;
	generate -. &nbsp;end&nbsp; .-> __end__;
	generate -.-> refuse;
	generate -.-> vision_escalate;
	grade -.-> generate;
	grade -.-> refuse;
	grade -.-> vision_escalate;
	ocr -.-> rag;
	ocr -.-> vision_input;
	rag -.-> grade;
	rag -.-> refuse;
	vision_escalate --> generate;
	vision_input --> rag;
	refuse --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 3. 멀티스텝 시나리오 결과 (§4)

정본 실행: `.venv/bin/python -m w11.week11_scenarios` → `data/week11_scenarios.json`.
아래 표는 그 로그를 그대로 읽는다 — "실제 흐름" 열은 agent가 기록한 `route_history`에서 나온다.

In [6]:
import pandas as pd

rows = json.loads(Path("data/week11_scenarios.json").read_text())
table = pd.DataFrame([{
    "시나리오": r["id"],
    "입력": r["input_type"],
    "기대 흐름": " → ".join(r["expected_flow"]),
    "실제 흐름": " → ".join(r["actual_flow"]),
    "일치": r["flow_match"],
    "fallback": len(r["fallback_history"]),
    "거절": r["refused"],
    "latency(s)": r["latency_s"],
} for r in rows])
table

,시나리오,입력,기대 흐름,실제 흐름,일치,fallback,거절,latency(s)
0,S1,image,ocr → rag → grade → generate,ocr → rag → grade → generate,True,0,False,35.49
1,S2,text,rag → grade → vision_escalate → generate,rag → grade → generate → vision_escalate → gen...,True,2,True,30.60
2,S3,image,ocr → vision_input → rag → grade,ocr → vision_input → rag → grade → vision_esca...,True,3,True,27.10
3,S4,text,rag → grade → vision_escalate → generate,rag → grade → generate → vision_escalate → gen...,True,2,True,30.05


In [7]:
# fallback 발동 기록과 답변 전문
for r in rows:
    print(f"── {r['id']} {r['name']}")
    for fb in r["fallback_history"] or ["(fallback 없음)"]:
        print(f"   {fb}")
    print(f"   answer: {r['answer'][:200]}")
    print()

── S1 happy path: 이미지 페이지 → OCR → 검색 → 답변
   (fallback 없음)
   answer: 정수기 필터의 교체 주기는 중금속9 흡착 필터는 6개월, 바이러스 클리어 필터는 12개월입니다. 필터 교체 주기는 4인 가정 하루 10 L 사용을 기준으로 하며, 필터의 수명은 수질, 수압, 계절, 지역에 따라 차이가 있을 수 있습니다. (출처: waterpurifier_simple p.18)

── S2 core: 텍스트 질문(IR-A1) → 캡션 RAG 근거 부족 → Vision 에스컬레이션
   F2: generate(ungrounded) (caption_hit=True) → vision_escalate([airpurifier_simple p.18])
   F3: ungrounded / insufficient evidence (escalation exhausted) → refusal
   answer: 죄송합니다. 제공된 매뉴얼에서 해당 질문에 대한 정보를 찾을 수 없습니다.

── S3 edge: OCR-dead 도면 페이지 + 매뉴얼 밖 질문 → F1 우회 → 근거 부족 → 거절
   F1: ocr unusable (conf=0.00, chars=0) → vision_input
   F2: grade(not_relevant) (caption_hit=True) → vision_escalate([vacuumcleaner_complex p.40])
   F3: ungrounded / insufficient evidence (escalation exhausted) → refusal
   answer: 죄송합니다. 제공된 매뉴얼에서 해당 질문에 대한 정보를 찾을 수 없습니다.

── S4 bonus: IR-A3 아이콘 모양 — 두 번째 F2 케이스 (캡션 상한 vs 질의측 vision)
   F2: generate(ungrounded) (caption_hit=True) → vision_escalate([airpurifier_simple 

In [8]:
# 시나리오 전체 재실행 (reranker/스토어 로드 + OpenAI 호출, 수 분 소요)
if RUN_LIVE:
    from w11.scenarios import run_scenarios
    run_scenarios()

## 4. 정리

결과 해석·회고·12주차 개선점은 `docs/week11_architecture.md`에 정리.